In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

TRAIN HYBRID MODEL

In [ ]:
hybrid_model = Ridge()
hybrid_model.fit(X_train_hybrid, y_train_esm)

SHAP EXPLAINABILITY ANALYSIS

In [ ]:
explainer = shap.Explainer(hybrid_model, X_train_hybrid)
shap_values = explainer(X_test_hybrid[:200])

SHAP VISUALIZATIONS

In [ ]:
# shap bar plot (global importance)
plt.figure()
shap.plots.bar(shap_values, show=False)
plt.savefig("results/figures/shap_bar.png", bbox_inches="tight")
plt.close()

# shap beeswarm plot (dist. of impact)
plt.figure()
shap.plots.beeswarm(shap_values, show=False)
plt.savefig("results/figures/shap_beeswarm.png", bbox_inches="tight")
plt.close()

print("SHAP plots saved.")

SHAP FEATURE IMPORTANCE ANALYSIS

In [ ]:
mean_shap = np.abs(shap_values.values).mean(axis=0)
top_features = np.argsort(mean_shap)[-10:]

SHAP FEATURE REMOVAL

In [ ]:
# remove top shap features from hybrid
X_train_removed = np.delete(X_train_hybrid, top_features, axis=1)
X_test_removed = np.delete(X_test_hybrid, top_features, axis=1)

# retrain model
removed_model = Ridge()
removed_model.fit(X_train_removed, y_train_esm)

removed_preds = removed_model.predict(X_test_removed)

removed_mae = mean_absolute_error(y_test_esm, removed_preds)
removed_r2 = r2_score(y_test_esm, removed_preds)

print("\nSHAP FEATURE REMOVAL RESULTS")
print("MAE:", removed_mae)
print("R2:", removed_r2)

BASELINE COMPARISON

In [ ]:
# full hybrid baseline model (no feature removal)
baseline_model = Ridge()
baseline_model.fit(X_train_hybrid, y_train_esm)

baseline_preds = baseline_model.predict(X_test_hybrid)

baseline_mae = mean_absolute_error(y_test_esm, baseline_preds)
baseline_r2 = r2_score(y_test_esm, baseline_preds)

print("\nBASELINE vs SHAP REMOVAL")
print("Baseline MAE:", baseline_mae)
print("Removed MAE:", removed_mae)

print("Baseline R2:", baseline_r2)
print("Removed R2:", removed_r2)

In [ ]:
RESULTS

In [ ]:
results = pd.DataFrame([
    {
        "Model": "Hybrid (Full)",
        "MAE": baseline_mae,
        "R2": baseline_r2
    },
    {
        "Model": "After SHAP Feature Removal",
        "MAE": removed_mae,
        "R2": removed_r2
    }
])

results.to_csv("results/shap_feature_removal_results.csv", index=False)

results